# Reproducing Davatolhagh et al. (2025) from dandiset 001712

**Paper.** Davatolhagh MF, Couto J, Melin M, Oesch LT, Findling C, Morales P, Hong I, International
Brain Laboratory, Churchland AK (2025). Altered use of prior expectations and modified neural
dynamics in a mouse model of autism. bioRxiv, version 1.
[doi:10.1101/2025.09.12.675910](https://doi.org/10.1101/2025.09.12.675910). This attempt used the
full text of the preprint (CC-BY-NC-ND).

**Dandiset.** [DANDI:001712](https://dandiarchive.org/dandiset/001712/draft), version `draft`. The
paper cites no version, and the dandiset has no published version: the archive lists only the
draft. The draft is used, with its modification time (2026-05-28T10:04:28Z) and the ids and sizes
of the three assets read, which the next cell checks against the archive. A draft can change; if
the check reports a difference, the numbers below may not match the attempt record.

**What the dandiset holds.** One session (2023-11-14) of one mouse, FD-28, in the biased
choice-world task with widefield imaging: a processed NWB file, a raw NWB file and three camera
videos. The paper analyses 11 wild-type and 8 Nrxn1a knockout mice with 10 sessions each. The
genotype of FD-28 is not stated in the files, the dandiset's metadata, or the public IBL
database. The findings that compare genotypes cannot be tested; those that the paper states for
mice of both genotypes are tested on this session.

**Code read.** The paper links `churchlandlab/chiCa` for its encoding analyses (read at commit
d98b3c6) and `jcouto/wfield` for preprocessing (commit 0befe16). The dandiset links the conversion
code `catalystneuro/IBL-widefield-to-nwb` (commit 4a6fb06). Choices that follow them are given in
the helper module `wfprior.py`.

| Claim | Data | Status |
|---|---|---|
| fig1c-psychometric: rightward choices rise with right contrast | dandiset | tested |
| fig1d-psychometric-params-genotype | dandiset | blocked: data missing |
| fig1e-rt-contrast: longer reaction times at lower contrast | dandiset | tested |
| fig1f-rt-genotype | dandiset | blocked: data missing |
| fig1g-learning-genotype | dandiset | blocked: data missing |
| fig2bc-prior-shift: more rightward choices in right-likely blocks | dandiset | tested |
| fig2bd-ko-smaller-shift | dandiset | blocked: data missing |
| fig2ef-block-adaptation: choices adapt after a block switch | dandiset | tested |
| fig2g-ko-slower-adaptation | dandiset | blocked: data missing |
| fig3b-contralateral-visual: lateralized visual responses | dandiset | tested |
| fig3c-ko-elevated-activity | dandiset | blocked: data missing |
| fig3d-ko-higher-correlation | dandiset | blocked: data missing |
| fig3e-ko-higher-variance | dandiset | blocked: data missing |
| fig4c-full-cvr2: the encoding model explains over half the variance | dandiset | tested |
| fig4cd-movement-dominates: movement predictors dominate | dandiset | tested |
| fig4cd-ko-more-movement | dandiset | blocked: data missing |
| figs2-ko-elevated-motion | dandiset | blocked: data missing |
| figs3-ko-higher-unique | dandiset | blocked: data missing |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats

import wfprior as wf

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
info = wf.draft_info()
print(f"dandiset {wf.DANDISET_ID} version {wf.DANDISET_VERSION}, modified {info['modified']}")
for path, (aid, size) in sorted(info["assets"].items()):
    print(f"  {size / 1e9:7.2f} GB  {aid}  {path}")
diff = wf.check_assets(info)
print("Assets differ from those this entry was written against:" if diff else
      "The assets read match those this entry was written against.", *diff, sep="\n")

f = wf.open_nwb(wf.PROCESSED)
trials = wf.load_trials(f)
svd = wf.load_svd(f)
frame_t = svd["t"]
movement = wf.load_movement(f, frame_t)
print(f"loaded trials, imaging components and movement in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
meta = {k: wf.text(f, "general/" + k) for k in ("lab", "institution", "protocol", "session_id", "keywords")}
subj = {k: wf.text(f, "general/subject/" + k) for k in ("subject_id", "sex", "strain", "date_of_birth", "species")}
print("Session:", wf.text(f, "session_start_time"), "| lab:", meta["lab"], "| protocol:", meta["protocol"])
print("Subject:", subj)
print("Subject fields:", sorted(f["general/subject"].keys()), "(no genotype field)")
print("Keywords in the file:", meta["keywords"])
for plane in ("ImagingPlaneCalcium", "ImagingPlaneIsosbestic"):
    p = f["general/optophysiology/" + plane]
    print(plane, "| indicator:", wf.text(f, f"general/optophysiology/{plane}/indicator"),
          "| excitation:", p["excitation_lambda"][()], "nm | location:", wf.text(f, f"general/optophysiology/{plane}/location"))
print("Assets: files per subject", pd.Series([p.split("/")[0] for p in info["assets"]]).value_counts().to_dict())

One subject and one session. The subject record has no genotype, and its description is the
paper's general description of the cross (KO and wild-type littermates). The paper (Methods) and
the file's keywords give the indicator as GCaMP6s, while both imaging planes say GCaMP6f. The
dandiset's own keywords list "CNTNALP2", a different gene from the Nrxn1a of the paper and the
file (see the summary).

In [ ]:
print("Trials:", len(trials))
for c in ["gabor_stimulus_contrast", "gabor_stimulus_side", "mouse_wheel_choice", "block_type", "probability_left",
          "is_mouse_rewarded", "reward_volume_uL"]:
    print(f"  {c}: {trials[c].value_counts(dropna=False).sort_index().to_dict()}")
print("Blocks:", trials.block_index.nunique(), "| first block length:", int((trials.block_index == 0).sum()),
      "| biased block lengths:", trials[trials.block_index > 0].groupby("block_index").size().describe()[["min", "50%", "max"]].to_dict())
side_given_block = pd.crosstab(trials.block_type, trials.gabor_stimulus_side, normalize="index").round(3)
print("Stimulus side by block (fraction):", side_given_block.to_dict("index"))
ok = trials.mouse_wheel_choice != "none"
correct = ((trials.choice_right == 1) & (trials.gabor_stimulus_side == "right")) | ((trials.choice_right == 0) & (trials.gabor_stimulus_side == "left"))
zero = trials.gabor_stimulus_contrast == 0
print("Rewarded equals correct choice (counter-clockwise = right) on non-zero contrast trials:",
      bool((trials.is_mouse_rewarded[~zero] == correct[~zero]).all()),
      "| on 0% trials:", f"{(trials.is_mouse_rewarded[zero] == correct[zero]).mean():.3f}")
times = ["start_time", "quiescence_period", "gabor_stimulus_onset_time", "auditory_cue_time", "wheel_movement_onset_time",
         "choice_registration_time", "feedback_time", "gabor_stimulus_offset_time", "stop_time"]
print(trials[times].describe().T[["count", "min", "max"]].round(3).to_string())
d = trials.gabor_stimulus_onset_time - trials.auditory_cue_time
print(f"Stimulus onset minus go cue (s): median {d.median():.4f}, range {d.min():.4f} to {d.max():.4f}")
q = trials.gabor_stimulus_onset_time - trials.start_time
print(f"Stimulus onset minus trial start (s): median {q.median():.3f}; quiescence period range {trials.quiescence_period.min():.3f} to {trials.quiescence_period.max():.3f}")
print(f"Reaction time (s): {trials.rt.notna().sum()} finite, {(trials.rt <= 0).sum()} at or before stimulus onset, "
      f"median {trials.rt.median():.3f}, range {trials.rt.min():.3f} to {trials.rt.max():.3f}")
g = trials.wheel_movement_onset_time - trials.auditory_cue_time
print(f"Wheel movement onset minus go cue (s): {int((g < 0).sum())} of {int(g.notna().sum())} negative, minimum {g.min():.3f}")
print(f"Feedback minus choice registration (s): median {(trials.feedback_time - trials.choice_registration_time).median():.4f}")

The contrasts are in percent (0 to 100) and the five values of the description occur with similar
counts. The first block is the 90 unbiased trials the description gives, and the stimulus side
follows the block probabilities (about 0.8 on the likely side). A counter-clockwise turn is the
correct response to a right stimulus on every trial with a visible stimulus, which confirms the
coding of rightward choices. The go cue and the photodiode-detected stimulus onset are close to
simultaneous, as the description says (offsets printed above). The wheel movement onset is
described as the first movement after the go cue, but some onsets fall before the go cue and the
stimulus onset (counts above); those trials are left out of reaction-time measures.

In [ ]:
dt = np.diff(frame_t)
print(f"Imaging: {len(frame_t)} frames per channel, {frame_t[0]:.2f} to {frame_t[-1]:.2f} s, "
      f"interval median {np.median(dt) * 1000:.3f} ms ({1 / np.median(dt):.2f} Hz), range {dt.min() * 1000:.3f} to {dt.max() * 1000:.3f} ms")
print("Trials span", round(trials.start_time.min(), 2), "to", round(trials.stop_time.max(), 2), "s, inside the imaging")
raw = wf.open_nwb(wf.RAW)
raw_t = raw["acquisition/OnePhotonSeriesCalcium/timestamps"][:]
print("Raw 470 nm frame timestamps equal the processed component timestamps:", np.array_equal(raw_t, frame_t),
      f"(raw frames {raw['acquisition/OnePhotonSeriesCalcium/data'].shape})")
print("Raw imaging rate stated:", raw["general/optophysiology/ImagingPlaneCalcium/imaging_rate"][()], "Hz")
u = svd["u"].reshape(len(svd["u"]), -1).astype(np.float64)
gram = u @ u.T
print(f"Spatial components: {svd['u'].shape}, Gram diagonal {np.diag(gram).min():.5f} to {np.diag(gram).max():.5f}, "
      f"largest off-diagonal {np.abs(gram - np.diag(np.diag(gram))).max():.2e}, every pixel nonzero: {bool((np.abs(u).sum(0) > 0).all())}")
svt = svd["svt"]
recon = svt[::5000] @ u
print(f"Temporal components: {svt.shape}; reconstructed dF/F over {len(recon)} sampled frames: 1st to 99th percentile "
      f"{np.percentile(recon, 1):.4f} to {np.percentile(recon, 99):.4f}")
print(f"Mean image: {svd['mean_image'].shape}, range {svd['mean_image'].min():.0f} to {svd['mean_image'].max():.0f}")
print("Lab metadata groups in the processed file:", sorted(f["general"].keys()))

The Methods give 30 frames per second per channel; the timestamps give 31.2 Hz, with regular
intervals, and the raw file's imaging plane states 15 Hz. The raw and the processed file hold the same recording (same timestamps), so it is
counted once. The spatial components are orthonormal, so variance in pixel space is the sum of the
variances of the temporal components. They cover every pixel of the frame: there is no brain mask.
The conversion code's documentation says the processed file holds landmarks and a registration to
the Allen atlas (`lab_meta_data` `localization`, `atlas_registration`); neither file holds them, so
no pixel can be assigned to a cortical area. The reconstructed values are of the order of a few
percent, consistent with dF/F.

In [ ]:
print("Movement variables per imaging frame:")
print(movement.describe().T[["mean", "std", "min", "max"]].round(3).to_string())
for cam in ("Left", "Right", "Body"):
    t = f[f"processing/motion_energy/{cam}CameraMotionEnergy/timestamps"][:]
    print(f"{cam} camera: {len(t)} frames, {1 / np.median(np.diff(t)):.2f} Hz")
print("Units: wheel", f["processing/wheel/WheelVelocitySmoothed/data"].attrs["unit"],
      "| paws", f["processing/pose_estimation/LeftCamera/PoseEstimationSeriesLeftPaw/data"].attrs["unit"],
      "| motion energy", f["processing/motion_energy/BodyCameraMotionEnergy/data"].attrs["unit"])
for cam in ("Left", "Right"):
    for paw in ("LeftPaw", "RightPaw"):
        conf = f[f"processing/pose_estimation/{cam}Camera/PoseEstimationSeries{paw}/confidence"][:]
        print(f"  {cam} camera {paw}: fraction of frames with likelihood below 0.9: {(conf < 0.9).mean():.3f}")

The camera rates match the Methods (left 60 Hz, right 150 Hz, body 30 Hz). The ROI motion energy
descriptions give the ROI's position in pixels but not the body part; the paper's model uses
whisker-pad motion energy from the side cameras and a body ROI, and the IBL places its side-camera
ROI on the whisker pad, so these are taken to be the whisker pads and the body. That is an
assumption.

In [ ]:
# Clock checks. (1) The wheel turns the way the choice says, after the movement onset of the trial table.
wt = movement.wheel_velocity.to_numpy()
on = wf.frame_index(frame_t, trials.wheel_movement_onset_time.fillna(trials.choice_registration_time).to_numpy())
lags = np.arange(-15, 16)
vel = np.stack([wt[on + k] for k in lags])
sign = np.where(trials.choice_right == 1, 1, -1)
signed = (vel * sign[None, :])[:, ok.to_numpy()]
print("Wheel velocity (rad/s) times choice sign (+1 right), median over trials, by frame from movement onset:")
print(pd.Series(np.median(signed, axis=1), index=lags).round(3).to_string())
print("Wheel position reference frame:", wf.text(f, "processing/wheel/WheelPosition/reference_frame"))
after = vel[15 + 3:15 + 10][:, ok.to_numpy()].mean(0)
ccw = (trials.mouse_wheel_choice == "counter_clockwise").to_numpy()[ok.to_numpy()]
print(f"Mean wheel velocity 0.1 to 0.3 s after movement onset: negative on {(after[ccw] < 0).mean():.3f} of counter-clockwise choices, "
      f"positive on {(after[~ccw] > 0).mean():.3f} of clockwise choices")

# (2) Licks follow rewarded feedback.
licks = f["processing/lick_times/EventsLickTimes/timestamps"][:]
def lick_rate(t0, a, b):
    return np.array([((licks >= t + a) & (licks < t + b)).sum() / (b - a) for t in t0])
fb = trials.feedback_time.to_numpy()
rew = trials.is_mouse_rewarded.to_numpy()
for name, (a, b) in {"-1 to 0 s": (-1, 0), "0 to 1 s": (0, 1)}.items():
    print(f"Lick rate {name} from feedback: rewarded {lick_rate(fb[rew], a, b).mean():.2f}/s, "
          f"unrewarded {lick_rate(fb[~rew], a, b).mean():.2f}/s")

In [ ]:
# (3) Imaging and the trial table. Two contrasts, each as the norm in pixel space of the difference between the mean
# responses of two trial groups, by frame from the event, baseline -0.6 to -0.3 s before it. Null: the labels permuted
# within strata (200 permutations), which keeps any difference that comes from the strata.
#   feedback: rewarded against unrewarded trials, strata = absolute contrast;
#   stimulus: right against left stimulus at 25 and 100% contrast, strata = block type.
strong = (trials.gabor_stimulus_contrast >= 25).to_numpy()
right = (trials.gabor_stimulus_side == "right").to_numpy()
onset = wf.frame_index(frame_t, trials.gabor_stimulus_onset_time.to_numpy())
feedback = wf.frame_index(frame_t, trials.feedback_time.to_numpy())
fps = 1 / np.median(np.diff(frame_t))
clk_lags = np.arange(-19, 17)

def clock_check(event_frames, label, strata, rng):
    seg = np.stack([svt[o + clk_lags] - svt[o - 19:o - 9].mean(0) for o in event_frames])
    def course(lab):
        return np.linalg.norm(seg[lab].mean(0) - seg[~lab].mean(0), axis=1)
    null = []
    for _ in range(200):
        perm = label.copy()
        for s_ in np.unique(strata):
            m = strata == s_
            perm[m] = rng.permutation(label[m])
        null.append(course(perm))
    out = pd.DataFrame({"s": np.round(clk_lags / fps, 3), "true": course(label), "null_95th": np.percentile(null, 95, axis=0)}).set_index("s")
    above = out.true > out.null_95th
    run = above & above.shift(-1, fill_value=False) & above.shift(-2, fill_value=False)
    first = out.index[(out.index > -0.3) & run.to_numpy()]
    return out, int(above[out.index < 0].sum()), (first[0] if len(first) else None)

rng_clk = np.random.default_rng(7)
fb_course, fb_pre, fb_first = clock_check(feedback, trials.is_mouse_rewarded.to_numpy(), trials.gabor_stimulus_contrast.to_numpy(), rng_clk)
st_course, st_pre, st_first = clock_check(onset[strong], right[strong], trials.block_type.to_numpy()[strong], rng_clk)
both = pd.concat({"feedback: rewarded vs not": fb_course, "stimulus: right vs left": st_course}, axis=1)
print(f"Feedback: {int(trials.is_mouse_rewarded.sum())} rewarded, {int((~trials.is_mouse_rewarded).sum())} unrewarded trials; "
      f"stimulus: {int((strong & right).sum())} right, {int((strong & ~right).sum())} left trials")
print(both.round(3).to_string())
for name, pre, first in (("feedback", fb_pre, fb_first), ("stimulus", st_pre, st_first)):
    print(f"{name}: frames before the event above the null 95th percentile {pre} of {int((clk_lags < 0).sum())}; "
          f"first of three consecutive frames above it at {first} s")
lick_lat = [licks[(licks > t) & (licks < t + 1)].min() - t for t in fb[rew] if ((licks > t) & (licks < t + 1)).any()]
print(f"First lick after rewarded feedback (video clock): median {np.median(lick_lat):.3f} s over {len(lick_lat)} trials")
rts = trials.rt[strong & ok.to_numpy() & (trials.rt > 0).to_numpy()]
print(f"Movement onset after the stimulus on the stimulus-contrast trials: median {rts.median():.3f} s")

The wheel starts to turn at the trial table's movement onset, and licking rises after rewarded
feedback and not after unrewarded feedback, so the wheel, the lick times and the trial table share
a clock. The direction disagrees with the descriptions: the wheel series say counter-clockwise is
positive, but trials labelled `counter_clockwise` turn the wheel with negative velocity. The choice
labels agree with the rewards (above), so one of the two sign conventions is misdescribed; the
analysis uses the choice labels.

For both contrasts the difference stays below the null's 95th percentile on every frame before
the event (0 of 19). After feedback the rewarded-unrewarded difference exceeds the null from 0.224
s, close to the median first lick after a reward on the video clock (0.216 s), as expected if
licking drives the cortical signal and the imaging, the cameras and the trial table share a clock.
After the stimulus the right-left difference exceeds the null only from 0.256 s, later than a
visual latency; on these trials the median movement onset is 0.086 s, so the side difference may
come from the choice movement. The checks are consistent with a shared clock to within a few
frames; they do not show a visual response at its expected latency.

## fig1c-psychometric

> Mice of both genotypes show sigmoidal psychometric curves, with the proportion of rightward
> choices increasing as the stimulus contrast increases on the right.

In [ ]:
ch = trials[ok]
psy = ch.groupby("signed_contrast").choice_right.agg(["mean", "size"]).rename(columns={"mean": "p_right", "size": "trials"})
print(psy.round(3).to_string())
rho, p_rho = stats.spearmanr(psy.index, psy.p_right)
span = psy.p_right.loc[100] - psy.p_right.loc[-100]
print(f"Spearman rho of P(right) with signed contrast over {len(psy)} levels: {rho:.3f} (p = {p_rho:.2g})")
print(f"P(right) at +100% minus at -100%: {span:.3f}")
fit_all = wf.fit_psychometric(ch.signed_contrast.to_numpy(), ch.choice_right.to_numpy())
print("Lapse-and-erf fit, all choice trials:", {k: round(float(v), 3) for k, v in fit_all.items()})
rng = np.random.default_rng(101)
boot = []
for _ in range(200):
    s = ch.sample(len(ch), replace=True, random_state=rng.integers(1 << 31))
    boot.append(wf.fit_psychometric(s.signed_contrast.to_numpy(), s.choice_right.to_numpy())["sigma"])
print(f"sigma (contrast %, inverse slope): {fit_all['sigma']:.2f}, bootstrap 95% interval {np.percentile(boot, 2.5):.2f} to {np.percentile(boot, 97.5):.2f}")
print("By block (variant):")
for b, g in ch.groupby("block_type"):
    r = wf.fit_psychometric(g.signed_contrast.to_numpy(), g.choice_right.to_numpy())
    print(f"  {b}: n = {len(g)}, P(right) at -100 / +100: {g[g.signed_contrast == -100].choice_right.mean():.3f} / "
          f"{g[g.signed_contrast == 100].choice_right.mean():.3f}, bias {r['bias']:.2f}, sigma {r['sigma']:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
cc = np.linspace(-100, 100, 201)
ax.plot(psy.index, psy.p_right, "ko")
ax.plot(cc, wf.psychometric(cc, *[fit_all[k] for k in ("bias", "sigma", "lapse_left", "lapse_right")]), "k-")
ax.set_xlabel("signed contrast (%)")
ax.set_ylabel("P(rightward choice)")
ax.set_ylim(0, 1)
ax.set_title("FD-28, all blocks")
plt.show()
print("Plotted P(right) range:", round(psy.p_right.min(), 3), "to", round(psy.p_right.max(), 3), "; none outside the axes")

**Result: reproduced.** The proportion of rightward choices rises at every step of signed
contrast, from 0.112 at -100% to 1.000 at +100% (Spearman rho 1.000 over nine levels; span 0.887).
The lapse-and-erf fit gives sigma 17.04% contrast (bootstrap 95% interval 13.98 to 20.47), a bias
of -6.77% (toward rightward choices) and a left lapse of 0.116. This is one session of one mouse of
unknown genotype.

## fig1e-rt-contrast

> Reaction time (time to first movement) is longer at lower contrasts in both genotypes.

In [ ]:
rt_ok = ok & (trials.rt > 0)
r = trials[rt_ok]
print(f"Trials used: {len(r)} of {len(trials)} ({int((~ok).sum())} without a choice, {int((ok & ~(trials.rt > 0)).sum())} with no "
      "movement onset after the stimulus)")
tab = r.groupby("gabor_stimulus_contrast").rt.describe()[["count", "25%", "50%", "75%"]]
print(tab.round(3).to_string())
rho, p = stats.spearmanr(r.gabor_stimulus_contrast, r.rt)
print(f"Spearman rho of |contrast| with reaction time: {rho:.3f}, p = {p:.2g}")
print(f"Median at 0%: {tab.loc[0.0, '50%']:.3f} s; at 100%: {tab.loc[100.0, '50%']:.3f} s")
for name, g in (("correct", r[correct[rt_ok]]), ("incorrect", r[~correct[rt_ok]])):
    rr, pp = stats.spearmanr(g.gabor_stimulus_contrast, g.rt)
    print(f"Variant, {name} trials only: n = {len(g)}, rho {rr:.3f}, p = {pp:.2g}, median RT {g.rt.median():.3f} s")
for lim in (0.08, 0.15):
    g = r[r.rt > lim]
    rr, pp = stats.spearmanr(g.gabor_stimulus_contrast, g.rt)
    print(f"Variant, reaction times above {lim} s: n = {len(g)}, rho {rr:.3f}, p = {pp:.2g}")

**Result: not reproduced.** Over 750 trials the Spearman correlation between absolute
contrast and reaction time is -0.053 (p = 0.15), so the test of the criterion fails; the median at
0% (0.089 s) does exceed that at 100% (0.081 s). Reaction times are short at every contrast
(medians 0.081 to 0.095 s), and 161 trials have no movement onset after the stimulus. Keeping only
reaction times above 0.15 s (226 trials) gives a correlation of -0.226 (p = 0.00062), so the
expected relation appears among the slower responses; this variant does not change the outcome.

## fig2bc-prior-shift

> Mice use the block prior; rightward choices are more frequent in right-likely blocks than in
> left-likely blocks, most clearly when contrast is low, shifting the psychometric curve toward
> the likely side.

In [ ]:
biased = trials[ok & (trials.block_type != "unbiased")]
tab = biased.pivot_table(index="signed_contrast", columns="block_type", values="choice_right", aggfunc=["mean", "size"])
shift = pd.DataFrame({"p_right_left_block": tab[("mean", "left_block")], "p_right_right_block": tab[("mean", "right_block")],
                      "n_left_block": tab[("size", "left_block")], "n_right_block": tab[("size", "right_block")]})
shift["right_minus_left"] = shift.p_right_right_block - shift.p_right_left_block
print(shift.round(3).to_string())
z = biased[biased.signed_contrast == 0]
a = int(z[(z.block_type == "right_block")].choice_right.sum()); n_a = int((z.block_type == "right_block").sum())
b = int(z[(z.block_type == "left_block")].choice_right.sum()); n_b = int((z.block_type == "left_block").sum())
odds, p0 = stats.fisher_exact([[a, n_a - a], [b, n_b - b]], alternative="greater")
print(f"0% contrast: P(right) {a}/{n_a} = {a / n_a:.3f} in right-likely blocks, {b}/{n_b} = {b / n_b:.3f} in left-likely blocks; "
      f"difference {a / n_a - b / n_b:.3f}, one-sided Fisher p = {p0:.2g}")
fits = {bt: wf.fit_psychometric(g.signed_contrast.to_numpy(), g.choice_right.to_numpy()) for bt, g in trials[ok].groupby("block_type")}
print(pd.DataFrame(fits).T[["bias", "sigma", "lapse_left", "lapse_right", "n"]].astype(float).round(3).to_string())
print(f"Bias, left-likely minus right-likely block: {fits['left_block']['bias'] - fits['right_block']['bias']:.2f} % contrast")
low = biased[biased.gabor_stimulus_contrast <= 12.5]
for bt in ("right_block", "left_block"):
    g = low[low.block_type == bt]
    print(f"Contrast 12.5% or less, {bt}: P(right) {g.choice_right.mean():.3f} (n = {len(g)})")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 3.5))
for bt, colour in (("left_block", "tab:blue"), ("right_block", "tab:orange"), ("unbiased", "0.5")):
    g = trials[ok & (trials.block_type == bt)].groupby("signed_contrast").choice_right.mean()
    ax.plot(g.index, g.values, "o", color=colour, label=bt)
    ax.plot(cc, wf.psychometric(cc, *[fits[bt][k] for k in ("bias", "sigma", "lapse_left", "lapse_right")]), color=colour)
ax.set_xlabel("signed contrast (%)")
ax.set_ylabel("P(rightward choice)")
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
plt.show()
print("All plotted proportions lie in 0 to 1; none outside the axes")

**Result: reproduced.** At 0% contrast the mouse chose right on 0.831 of trials (74 of 89)
in right-likely blocks and 0.507 (37 of 73) in left-likely blocks, a difference of 0.325
(one-sided Fisher p = 9.2e-06). The difference is 0.182 to 0.458 at every signed contrast except
+100%, where both blocks give 1.000, and the psychometric bias moves by 19.41% contrast between
left-likely (2.86) and right-likely (-16.54) blocks. Whether the shift is smaller than in
wild-type mice cannot be tested on one mouse of unknown genotype.

## fig2ef-block-adaptation

> After a block switch, mice adapt their choices toward the newly likely side, and the proportion
> of choices toward the block side rises from an initial value A to an asymptote B with a time
> constant tau.

In [ ]:
def switch_curve(sel):
    rows = []
    for bi in sorted(trials.block_index.unique()):
        if bi < 2:   # switches between biased blocks: from block 1 to 2 onward
            continue
        g = trials[(trials.block_index == bi)]
        toward_right = g.block_type.iloc[0] == "right_block"
        for k, (i, t) in enumerate(g.iterrows()):
            if k >= 40:
                break
            if not np.isfinite(t.choice_right) or not sel(t):
                continue
            rows.append({"block": bi, "t": k, "toward": float(t.choice_right == (1.0 if toward_right else 0.0))})
    return pd.DataFrame(rows)

def model(t, A, B, tau):
    return B - (B - A) * np.exp(-t / tau)

def adaptation(df, label):
    curve = df.groupby("t").toward.mean()
    (A, B, tau), cov = optimize.curve_fit(model, curve.index.to_numpy(float), curve.to_numpy(), p0=[0.5, 0.8, 5],
                                          bounds=([0, 0, 0.01], [1, 1, 200]))
    early = df[df.t <= 4].toward; late = df[(df.t >= 20) & (df.t <= 39)].toward
    _, p = stats.fisher_exact([[late.sum(), len(late) - late.sum()], [early.sum(), len(early) - early.sum()]], alternative="greater")
    print(f"{label}: {df.block.nunique()} switches, {len(df)} trials; fit A = {A:.3f}, B = {B:.3f}, tau = {tau:.2f} trials "
          f"(SE {np.sqrt(np.diag(cov))[2]:.2f}); toward block side trials 0-4 {early.mean():.3f} (n = {len(early)}), "
          f"trials 20-39 {late.mean():.3f} (n = {len(late)}), one-sided Fisher p = {p:.2g}")
    return curve, (A, B, tau)

sw_all = switch_curve(lambda t: True)
print("Block lengths of the switches used:", trials[trials.block_index >= 2].groupby("block_index").size().tolist())
curve_all, par_all = adaptation(sw_all, "All contrasts")
curve_low, par_low = adaptation(switch_curve(lambda t: t.gabor_stimulus_contrast <= 12.5), "Variant, contrast 12.5% or less")
print("Per switch, fraction toward the block side in trials 0-4 and 20-39:")
print(sw_all.assign(phase=np.where(sw_all.t <= 4, "0-4", np.where(sw_all.t >= 20, "20-39", "5-19")))
      .pivot_table(index="block", columns="phase", values="toward", aggfunc="mean").round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
tt = np.arange(40)
ax.plot(curve_all.index, curve_all.values, "ko", ms=3, label="all contrasts")
ax.plot(tt, model(tt, *par_all), "k-")
ax.plot(curve_low.index, curve_low.values, "o", color="tab:green", ms=3, label="contrast <= 12.5%")
ax.plot(tt, model(tt, *par_low), color="tab:green")
ax.set_xlabel("trials after block switch")
ax.set_ylabel("P(choice toward block side)")
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
plt.show()
print("Plotted proportions range", round(min(curve_all.min(), curve_low.min()), 3), "to", round(max(curve_all.max(), curve_low.max()), 3))

**Result: reproduced.** Over 14 switches between biased blocks, the proportion of choices
toward the new block side is 0.543 in trials 0 to 4 and 0.827 in trials 20 to 39 (one-sided
Fisher p = 2.7e-06). The fit gives A = 0.480, B = 1.000 (at its upper bound) and tau = 25.12
trials with a standard error of 16.14, so the time constant is poorly determined in one session.
At contrasts of 12.5% or less the pattern is the same (0.452 against 0.820; tau 21.57).

## fig3b-contralateral-visual

> Visual cortex (VISp) responds robustly to contralateral visual stimuli.

The files carry no atlas registration, so the test is in image coordinates (see the claims file).

In [ ]:
mean_img = svd["mean_image"]
H, W = mean_img.shape
cols = np.arange(int(W * 0.3), int(W * 0.7))
midline = int(cols[np.argmin(mean_img[:, cols].mean(0))])
print(f"Midline estimate: column {midline} of {W} (lowest mean fluorescence in columns {cols[0]} to {cols[-1]})")
ev = wf.evoked(svt, onset)                       # trials x components
uim = svd["u"].reshape(len(svd["u"]), -1).astype(np.float64)

def peak_test(sel_trials):
    rmap = (ev[sel_trials & strong & right].mean(0) - ev[sel_trials & strong & ~right].mean(0)) @ uim
    res = {}
    for name, m in (("right_minus_left", rmap), ("left_minus_right", -rmap)):
        p = int(np.argmax(m)); y, x = divmod(p, W)
        res[name] = {"row": y, "col": x, "side": "left of midline" if x < midline else "right of midline", "value": m[p], "pixel": p}
    return rmap, res

all_trials = np.ones(len(trials), bool)
rmap, peaks = peak_test(all_trials)
for name, r in peaks.items():
    vals = ev @ uim[:, r["pixel"]]
    a, b = (vals[strong & right], vals[strong & ~right]) if name == "right_minus_left" else (vals[strong & ~right], vals[strong & right])
    _, p = stats.mannwhitneyu(a, b, alternative="greater")
    print(f"{name}: peak at row {r['row']}, column {r['col']} ({r['side']}), difference {r['value']:.4f} dF/F; "
          f"defining side {np.median(a):.4f} against other side {np.median(b):.4f} (median dF/F), one-sided Mann-Whitney p = {p:.2g}")
print("Peaks on opposite sides of the midline:", peaks["right_minus_left"]["side"] != peaks["left_minus_right"]["side"])
print(f"Mean image at the two peaks: {mean_img.ravel()[peaks['right_minus_left']['pixel']]:.0f} and "
      f"{mean_img.ravel()[peaks['left_minus_right']['pixel']]:.0f}; median over the frame {np.median(mean_img):.0f}, 5th percentile {np.percentile(mean_img, 5):.0f}")
for side, sel in (("right", strong & right), ("left", strong & ~right)):
    m = ev[sel].mean(0) @ uim
    p = int(np.argmax(m)); y, x = divmod(p, W)
    print(f"Peak of the {side}-stimulus evoked map itself: row {y}, column {x}, {m[p]:.4f} dF/F")

In [ ]:
# Variant: peaks chosen on odd trials, tested on even trials (the main test chooses and tests on the same trials).
idx = np.arange(len(trials))
_, pk_odd = peak_test(idx % 2 == 1)
for name, r in pk_odd.items():
    vals = ev @ uim[:, r["pixel"]]
    sel_even = idx % 2 == 0
    a, b = (vals[sel_even & strong & right], vals[sel_even & strong & ~right]) if name == "right_minus_left" else (vals[sel_even & strong & ~right], vals[sel_even & strong & right])
    _, p = stats.mannwhitneyu(a, b, alternative="greater")
    print(f"Odd-trial {name} peak: row {r['row']}, column {r['col']} ({r['side']}); even trials: {np.median(a):.4f} against {np.median(b):.4f}, p = {p:.2g}")
# Variant: correct trials only and 100% contrast only.
for label, sel in (("correct trials", correct.to_numpy()), ("100% contrast", (trials.gabor_stimulus_contrast == 100).to_numpy())):
    m = (ev[sel & strong & right].mean(0) - ev[sel & strong & ~right].mean(0)) @ uim
    pr, pl = int(np.argmax(m)), int(np.argmax(-m))
    print(f"Variant, {label}: right-minus-left peak column {pr % W}, left-minus-right peak column {pl % W} (midline {midline})")
# Variant: peaks only among pixels brighter than the median of the mean image and 20 pixels or more from the frame edge.
yy, xx = np.mgrid[:H, :W]
mask = ((mean_img > np.median(mean_img)) & (yy >= 20) & (yy < H - 20) & (xx >= 20) & (xx < W - 20)).ravel()
print(f"Variant mask: {mask.sum()} of {mask.size} pixels")
for name, m in (("right_minus_left", rmap), ("left_minus_right", -rmap)):
    p = int(np.flatnonzero(mask)[np.argmax(m[mask])]); y, x = divmod(p, W)
    vals = ev @ uim[:, p]
    a, b = (vals[strong & right], vals[strong & ~right]) if name == "right_minus_left" else (vals[strong & ~right], vals[strong & right])
    _, pv = stats.mannwhitneyu(a, b, alternative="greater")
    print(f"  masked {name}: peak row {y}, column {x} ({'left' if x < midline else 'right'} of midline), {m[p]:.4f} dF/F; "
          f"{np.median(a):.4f} against {np.median(b):.4f}, p = {pv:.2g}")
fig, axs = plt.subplots(1, 3, figsize=(12, 3.2))
axs[0].imshow(mean_img, cmap="gray"); axs[0].axvline(midline, color="y"); axs[0].set_title("mean image, midline")
lim = np.percentile(np.abs(rmap), 99.5)
im = axs[1].imshow(rmap.reshape(H, W), cmap="RdBu_r", vmin=-lim, vmax=lim); axs[1].axvline(midline, color="k", lw=0.5)
for name, r in peaks.items():
    axs[1].plot(r["col"], r["row"], "k+" if name == "right_minus_left" else "kx", ms=10)
axs[1].set_title("right minus left stimulus, 0.1-0.4 s"); plt.colorbar(im, ax=axs[1], fraction=0.04)
pix = peaks["right_minus_left"]["pixel"]; pix2 = peaks["left_minus_right"]["pixel"]
for p_, lab in ((pix, "+ peak"), (pix2, "x peak")):
    for sel, ls, stim in ((strong & right, "-", "right"), (strong & ~right, "--", "left")):
        tr_ = np.stack([np.stack([svt[o + k] - svt[o - 9:o].mean(0) for k in lags]) @ uim[:, p_] for o in onset[sel]]).mean(0)
        axs[2].plot(lags / fps, tr_, ls, label=f"{lab}, {stim} stimulus")
axs[2].axvline(0, color="0.5"); axs[2].set_xlabel("s from stimulus onset"); axs[2].set_ylabel("dF/F"); axs[2].legend(fontsize=7)
plt.show()
print(f"Map colour range clipped at +/-{lim:.4f} dF/F (99.5th percentile of |value|); {(np.abs(rmap) > lim).sum()} pixels beyond it")

**Result: not reproduced.** Under the criterion fixed beforehand, the peaks of the
right-minus-left and left-minus-right maps both fall at the left edge of the frame (columns 0 and
2, rows 0 and 539), on the same side of the midline (column 316), with trial tests at p = 0.053
and 0.14. These corner pixels are darker in the mean image (534 and 926) than 95% of the frame (5th
percentile 1538), so they lie outside the imaged brain; the spatial components cover the whole
frame with no mask. In a variant restricted to pixels brighter than the median of the mean
image and 20 pixels from the edge, the right-minus-left peak lies left of the midline (column 199)
and the left-minus-right peak right of it (column 481), with p = 8.1e-15 and 1.6e-17. The variant
points to lateralized responses but does not change the outcome, and without an atlas registration
neither the hemisphere nor VISp can be named.

## Encoding model

The design follows the claims file and `wfprior.py`. The body camera's video and motion-energy
components are computed here by streaming the video from the archive.

In [ ]:
t0 = time.time()
vid = wf.body_video_components(f, frame_t)
print(f"Body camera: {vid['n_video_frames']} decoded frames, {vid['n_timestamps']} timestamps, {vid['pixels']} pixels after binning, "
      f"in {time.time() - t0:.0f} s; top {wf.N_VIDEO_COMPONENTS} components explain {vid['video_explained']:.3f} of video variance "
      f"and {vid['motion_explained']:.3f} of motion-energy variance")
ak = wf.fit_action_kernel(trials)
print(f"Action-kernel learning rate {ak['rate']:.2f} (negative log-likelihood {ak['nll']:.1f}; at rate 0.01 {ak['nll_grid'][0]:.1f}, at 0.99 {ak['nll_grid'][-1]:.1f}); "
      f"weights (bias, contrast, sqrt contrast, prior) {np.round(ak['weights'], 2).tolist()}")
print(f"Correlation of subjective prior with block (0.5 - probability_left): {np.corrcoef(ak['prior'], 0.5 - trials.probability_left)[0, 1]:.3f}")
valid = ok.to_numpy()
rows_frame, rows_tp, rows_trial = wf.segment_frames(trials, frame_t, valid)
points = wf.STIM_WINDOW[1] - wf.STIM_WINDOW[0] + wf.FEEDBACK_WINDOW[1] - wf.FEEDBACK_WINDOW[0]
mov_basic = movement.to_numpy()
mov_full = np.hstack([mov_basic, vid["video"], vid["motion"]])
X, index, analog = wf.build_design(trials, rows_trial, rows_tp, points, ak["prior"], mov_full, rows_frame)
Y = svt[rows_frame]
task_cols = np.concatenate([index[k] for k in ("choice", "stim_left", "stim_right", "outcome", "true_prior", "subjective_prior")])
mov_cols = index["movement_columns"]
print(f"Valid trials {valid.sum()} of {len(trials)}; rows {X.shape[0]} ({points} frames per trial); columns {X.shape[1]}: "
      f"intercept {len(index['intercept'])}, task {len(task_cols)}, movement {len(mov_cols)} "
      f"({mov_basic.shape[1]} basic + {vid['video'].shape[1]} video + {vid['motion'].shape[1]} motion energy)")
print(f"Rows in which a frame appears in both windows of a trial: {len(rows_frame) - len(np.unique(np.stack([rows_frame, rows_trial]), axis=1)[0])}")
u_flat = svd["u"][:, ::2, ::2].reshape(len(svd["u"]), -1).T.astype(np.float64)
print(f"Per-pixel cvR2 over every second pixel in each direction: {u_flat.shape[0]} pixels")

In [ ]:
t0 = time.time()
rng_fit = np.random.default_rng(404)
test_folds = wf.folds(rows_tp, rng_fit)
all_cols = np.concatenate([task_cols, mov_cols])
models = {
    "full": [], "task shuffled": task_cols, "movement shuffled": mov_cols, "intercept only": all_cols,
}
fits = {name: wf.fit_model(X, Y, analog, test_folds, cols, np.random.default_rng(404 + i), u_flat) for i, (name, cols) in enumerate(models.items())}
# Variant: without the body video components (columns dropped).
keep = np.setdiff1d(np.arange(X.shape[1]), mov_cols[mov_basic.shape[1]:])
Xb = X[:, keep]
pos = {c: i for i, c in enumerate(keep)}
mov_b = np.array([pos[c] for c in mov_cols[:mov_basic.shape[1]]]); task_b = np.array([pos[c] for c in task_cols])
an_b = [pos[c] for c in analog if c in pos]
fits["no video: full"] = wf.fit_model(Xb, Y, an_b, test_folds, [], np.random.default_rng(420), u_flat)
fits["no video: movement shuffled"] = wf.fit_model(Xb, Y, an_b, test_folds, mov_b, np.random.default_rng(421), u_flat)
fits["no video: task shuffled"] = wf.fit_model(Xb, Y, an_b, test_folds, task_b, np.random.default_rng(422), u_flat)
readings = {"a pooled pixel space": "pooled", "b mean over pixels": "pixel_mean", "c mean over components": "component_mean"}
table = pd.DataFrame({name: {r: fit[k] for r, k in readings.items()} for name, fit in fits.items()}).T
print(f"cvR2 by model and reading, mean over {wf.N_FOLDS} folds ({time.time() - t0:.0f} s):")
print(table.round(4).to_string())
print("Fold range of the full model:", {r: (round(fits['full']['folds'][k].min(), 4), round(fits['full']['folds'][k].max(), 4)) for r, k in readings.items()})
print("Median ridge penalty per model:", {k: round(v["alpha_median"], 1) for k, v in fits.items()})

## fig4c-full-cvr2

> The full linear encoding model explains more than half of single-trial cortical variance (cvR2
> 0.60 +/- 0.02 in KO and 0.54 +/- 0.02 in WT).

In [ ]:
lo, hi = 0.573 - 0.087, 0.573 + 0.087
rows = []
for r in readings:
    full, icpt = table.loc["full", r], table.loc["intercept only", r]
    rows.append({"reading": r, "full": full, "intercept only": icpt, "full minus intercept": full - icpt,
                 "full in range": lo <= full <= hi, "minus intercept in range": lo <= full - icpt <= hi})
cv = pd.DataFrame(rows).set_index("reading")
print(f"Tolerance range {lo:.3f} to {hi:.3f}")
print(cv.round(4).to_string())
print("Variant without body video components:")
print(pd.DataFrame({"full": table.loc["no video: full"], "minus intercept": table.loc["no video: full"] - table.loc["intercept only"]}).round(4).to_string())
pm = fits["full"]["pixel_map"]
print(f"Per-pixel cvR2 of the full model: median {np.median(pm):.3f}, 5th to 95th percentile {np.percentile(pm, 5):.3f} to {np.percentile(pm, 95):.3f}")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
im = axs[0].imshow(pm.reshape(H // 2, W // 2), vmin=0, vmax=1, cmap="viridis"); plt.colorbar(im, ax=axs[0], fraction=0.04)
axs[0].set_title("full model cvR2 per pixel")
im = axs[1].imshow((pm - fits["intercept only"]["pixel_map"]).reshape(H // 2, W // 2), vmin=0, vmax=1, cmap="viridis")
plt.colorbar(im, ax=axs[1], fraction=0.04); axs[1].set_title("full minus intercept only")
plt.show()
d = pm - fits["intercept only"]["pixel_map"]
print(f"Pixels outside 0 to 1: full {int(((pm < 0) | (pm > 1)).sum())}, difference {int(((d < 0) | (d > 1)).sum())} of {pm.size}")

**Result: partially reproduced.** The full model's cvR2 is 0.471 pooled in pixel space, 0.422
as the mean over pixels and 0.126 as the mean over components; with the intercept-only model
subtracted, 0.131, 0.142 and 0.087. None is within 0.486 to 0.660; reading (a) is above 0.3, which
the criterion counts as partial. The model lacks the video components of the left and right
cameras, which the paper includes. Without the body camera's components the pooled cvR2 falls from
0.471 to 0.414, so the missing cameras may account for part of the gap.

## fig4cd-movement-dominates

> Movement predictors account for most of the explained variance; shuffling them reduces cvR2 by
> much more than shuffling task predictors (delta R2 for movement 0.40 +/- 0.02 in KO and 0.33 +/-
> 0.02 in WT; task shuffle beta -0.02).

In [ ]:
lo, hi = 0.3685 - 0.0715, 0.3685 + 0.0715
rows = []
for r in readings:
    dm = table.loc["full", r] - table.loc["movement shuffled", r]
    dt_ = table.loc["full", r] - table.loc["task shuffled", r]
    k = readings[r]
    fm = fits["full"]["folds"][k] - fits["movement shuffled"]["folds"][k]
    ft = fits["full"]["folds"][k] - fits["task shuffled"]["folds"][k]
    rows.append({"reading": r, "movement dR2": dm, "task dR2": dt_, "movement in range": lo <= dm <= hi,
                 "task below movement/5": dt_ < dm / 5, "movement folds min": fm.min(), "task folds max": ft.max()})
dd = pd.DataFrame(rows).set_index("reading")
print(f"Tolerance range for movement dR2: {lo:.4f} to {hi:.4f}")
print(dd.round(4).to_string())
print("Variant without body video components:")
for r in readings:
    print(f"  {r}: movement dR2 {table.loc['no video: full', r] - table.loc['no video: movement shuffled', r]:.4f}, "
          f"task dR2 {table.loc['no video: full', r] - table.loc['no video: task shuffled', r]:.4f}")
print("Explained variance above intercept only, and the share left after each shuffle:")
for r in readings:
    above = table.loc["full", r] - table.loc["intercept only", r]
    print(f"  {r}: full minus intercept {above:.4f}; movement-shuffled minus intercept {table.loc['movement shuffled', r] - table.loc['intercept only', r]:.4f}; "
          f"task-shuffled minus intercept {table.loc['task shuffled', r] - table.loc['intercept only', r]:.4f}")

**Result: partially reproduced.** Shuffling the movement predictors lowers cvR2 by 0.104
(pooled), 0.119 (mean over pixels) and 0.067 (mean over components); shuffling the task predictors
lowers it by 0.017, 0.015 and 0.013. Movement dominates under every reading and in every fold
(smallest movement drop 0.065, largest task drop 0.019), but the movement drop is below the range
0.297 to 0.440 under every reading. The body camera's video components carry most of it: without
them the movement drop is 0.042 pooled against 0.021 for task, and 0.010 against 0.015 over
components.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| fig1c-psychometric | dandiset | reproduced |
| fig1d-psychometric-params-genotype | dandiset | blocked: data missing |
| fig1e-rt-contrast | dandiset | not reproduced |
| fig1f-rt-genotype | dandiset | blocked: data missing |
| fig1g-learning-genotype | dandiset | blocked: data missing |
| fig2bc-prior-shift | dandiset | reproduced |
| fig2bd-ko-smaller-shift | dandiset | blocked: data missing |
| fig2ef-block-adaptation | dandiset | reproduced |
| fig2g-ko-slower-adaptation | dandiset | blocked: data missing |
| fig3b-contralateral-visual | dandiset | not reproduced |
| fig3c-ko-elevated-activity | dandiset | blocked: data missing |
| fig3d-ko-higher-correlation | dandiset | blocked: data missing |
| fig3e-ko-higher-variance | dandiset | blocked: data missing |
| fig4c-full-cvr2 | dandiset | partially reproduced |
| fig4cd-movement-dominates | dandiset | partially reproduced |
| fig4cd-ko-more-movement | dandiset | blocked: data missing |
| figs2-ko-elevated-motion | dandiset | blocked: data missing |
| figs3-ko-higher-unique | dandiset | blocked: data missing |

**How the analysis differs from the paper's.** One session of one mouse, where the paper uses 10
sessions from each of 19 mice; the genotype is unknown, so values reported per genotype are
compared with a range spanning both groups. The encoding model has no video components from the left
and right cameras (left out for runtime), uses Lightning Pose paw speeds where the paper uses
DeepLabCut, takes the ROI motion energies to be the whisker pads and body, and uses an action-kernel
prior fitted here. The trial windows, the ridge penalty selection (generalized cross-validation) and
the three readings of cvR2 are choices made here, as the paper's widefield code is not public; the
lab's chiCa code sets the rest. VISp cannot be located without an atlas registration, so the visual
response is tested in image coordinates.

**What was found about the dandiset.** It holds one of the 17 sessions the conversion code lists
for 19 mice, and no genotype for its mouse. It has no published version. The atlas landmarks and
registration that the conversion documentation describes are not in either file. The imaging planes
give GCaMP6f where the paper and the file's keywords give GCaMP6s, the raw file states an imaging
rate of 15 Hz where the timestamps give 31.2 Hz per channel, the dandiset's keywords name
"CNTNALP2" where the paper and files name Nrxn1a, and the wheel's stated sign convention
(counter-clockwise positive) is the opposite of what the choice labels imply. The trial table, wheel,
licks and imaging are consistent with a shared clock.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")